# IMPORTS

In [ ]:
%load_ext autoreload

import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import numpy as np
import jax.numpy as jnp

import tol_colors as tc
import matplotlib.pyplot as plt
import plotting_tools.basic_plotting_tools as pts
colors = pts.set_plot_style(default_cmap=tc.sunset)
%matplotlib widget

In [ ]:
import differential_geometry.manifolds as manifolds
import plotting_tools.plot_manifolds as plot_manifolds

import differential_geometry.charts as charts
import plotting_tools.plot_charts as plot_charts

import differential_geometry.covectors as covectors

import differential_geometry.fields as fields
import plotting_tools.plot_fields as plot_fields

import differential_geometry.bundles as bundles
import plotting_tools.plot_bundles as plot_bundles

# 1D-mfd (ring) embedded in 3D

The base manifold is a ring $\mathcal{M} \cong S^1$ embedded in $\mathbb{R}^3$, so that
$d = 1$ and the total space of the tangent bundle has dimension $2d = 2$:

$$\left( T\mathcal{M}, \mathcal{O}_{T\mathcal{M}}, \mathcal{A}_{T\mathcal{M}} \right)_{2d}
\xrightarrow{\;\pi\;}
\left( \mathcal{M}, \mathcal{O}, \mathcal{A}_{C^\infty} \right)_{d}.$$

A point of the total space is the pair (base point ; components of the vector at that point), and
$\pi$ simply forgets the second block. To *draw* $T\mathcal{M}$ we must choose an ambient direction
along which each fibre is displayed — the bundle itself carries no such information. Taking the
constant direction $\hat{z}$, transverse to the plane of the ring, turns the total space into the
familiar cylinder, with the base manifold sitting inside it as the zero section. This picture is
faithful because $T S^1 \cong S^1 \times \mathbb{R}$ is trivializable; for a base manifold with a
non-trivial tangent bundle no such global picture exists.

The figure below assembles, on this example:

- the total space $T\mathcal{M}$ (cylinder) and the base $\mathcal{M}$ (zero section);
- the fibre $T_p\mathcal{M} = \mathrm{preim}_\pi(\{p\})$ over a point $p$, and the projection $\pi$;
- the region $T\mathcal{U} = \mathrm{preim}_\pi(\mathcal{U})$ over a chart domain;
- a vector field $\mathcal{X}$, drawn as what it is: a section, i.e. a curve inside $T\mathcal{M}$
  that meets each fibre exactly once;
- one tangent vector $v_{\gamma, p} = \mathcal{X}_p$ inside its fibre;
- and, on the right panel, the image of all of that under the bundle chart
  $\xi_{\mathscr{X}} : T\mathcal{U} \to \mathbb{R}^{2d}$, whose base block is the chart map
  $\mathscr{X}$ and whose fibre block is $(d\mathscr{X}^1)_p$.

In [ ]:
# === Base manifold: a ring of radius R embedded in R^3 ===
RADIUS = 1.0

def ring_embedding(params):
    """Phi: theta -> (R cos theta, R sin theta, 0)."""
    angle = params[0]
    return jnp.stack([RADIUS * jnp.cos(angle), RADIUS * jnp.sin(angle), 0.0 * angle])

manifold = manifolds.Manifold(
    name=r"$\mathcal{M}$", dim=1, ambient_dim=3, embedding_func=ring_embedding)

# === Chart (U, X): stereographic-like chart of the circle, X(theta) = tan(theta / 2) ===
theta_min, theta_max = -0.25, 1.15

def chart_map_U(params):
    return jnp.tan(params / 2.0)

def inverse_chart_map_U(chart_coords):
    return 2.0 * jnp.arctan(chart_coords)

chart_U = charts.Chart(
    name="U", manifold=manifold, boundary=(theta_min, theta_max),
    chart_map=chart_map_U, inverse_chart_map=inverse_chart_map_U)

# === A second chart (V, Y), used only for the transition-map check below ===
def chart_map_V(params):
    return params + 0.4 * jnp.sin(params)

chart_V = charts.Chart(
    name="V", manifold=manifold, boundary=(-0.6, 1.5), chart_map=chart_map_V)

# === A vector field on the ring (nowhere vanishing: T S^1 is trivializable) ===
def field_components(params):
    """X: components w.r.t. the parameter-space basis d/d(theta)."""
    return jnp.array([0.55 + 0.42 * jnp.sin(2.0 * params[0])])

field_X = fields.VectorField(manifold=manifold, vector_function=field_components)

# === The tangent bundle, with the fibres drawn along the constant direction z ===
bundle = bundles.TangentBundle(
    manifold=manifold, fibre_direction=jnp.array([0.0, 0.0, 1.0]), fibre_scale=1.0,
    name=r"$T\mathcal{M}$")

# === The base point p and the vector the field picks there ===
p_param = jnp.array([0.45])
p_component = float(field_X.evaluate_in_param_space(p_param)[0])

fibre_range = (-1.25, 1.25)      # window of fibre components drawn on the cylinder

## The bundle chart $\xi_{\mathscr{X}}$ and its transition map

$$\xi_{\mathscr{X}}(v_{\gamma, p}) = \big( \mathscr{X}^i(p) \,;\, (d\mathscr{X}^j)_p(v_{\gamma, p}) \big),
\qquad
\left( \xi_{\mathscr{Y}} \circ \xi_{\mathscr{X}}^{-1} \right)(\alpha ; \beta)
= \big( \mathscr{T}_{\mathcal{V}\mathcal{U}}(\alpha) \,;\, (J_{\mathcal{V}\mathcal{U}})^j_{\ k}(p)\, \beta^k \big).$$

In [ ]:
# === One point of the total space: the vector the field picks at p ===
bundle_point = bundle.fibre(p_param, jnp.array([p_component]))

print("=== a point of TM ===")
print("(theta ; u)          =", bundle_point)
print("pi(theta ; u)        =", bundle.projection(bundle_point), "   vs  p =", p_param)
print("section of the field =", bundle.section_of_field(field_X, p_param[None, :]))

print("\n=== bundle chart ===")
chart_point = bundle.chart_map(chart_U, bundle_point)
print("xi_X(v)              =", chart_point)
print("  base block  X(p)   =", chart_U.map_to_chart(p_param))
print("  fibre block X^1(p) =", field_X.components_in_chart(chart_U, p_param))
print("xi_X^{-1}(xi_X(v))   =", bundle.inverse_chart_map(chart_U, chart_point), " vs ", bundle_point)

print("\n=== transition map between bundle charts ===")
transition_point = bundle.chart_transition(chart_V, chart_U, chart_point)
J_VU = covectors.chart_transition_jacobian(chart_V, chart_U, p_param)
print("(xi_Y o xi_X^{-1})(alpha ; beta) =", transition_point)
print("  base block   Y(p)             =", chart_V.map_to_chart(p_param))
print("  fibre block  (J_VU)^j_k beta^k =", J_VU @ chart_point[0, 1:])

print("\n=== the section really is a section: pi o X = identity ===")
sample_points = jnp.linspace(-jnp.pi, jnp.pi, 7).reshape(-1, 1)
print("max |pi(X(p)) - p| =",
      float(jnp.max(jnp.abs(bundle.projection(bundle.section_of_field(field_X, sample_points))
                            - sample_points))))

## plot the bundle

Colours follow the hand-drawn figure: cyan for the total space, purple for the base manifold
(the zero section), red for everything chart-related, green for the fibre and the projection,
magenta for the section and blue for the single tangent vector. The two curved red arrows are the
maps themselves, anchored to the objects they act on (the anchors are computed after a first draw,
which is why `fig.canvas.draw()` is called before adding them).

In [ ]:
color_total_space = "paleturquoise"
color_base = "rebeccapurple"
color_chart = "red"
color_fibre = "green"
color_field = "magenta"
color_vector = "blue"

fig = plt.figure(figsize=(16, 7.5))
grid = fig.add_gridspec(1, 2, width_ratios=[1.25, 1.0], wspace=0.02,
                        left=0.01, right=0.98, bottom=0.05, top=0.95)
ax3d = fig.add_subplot(grid[0, 0], projection="3d")
ax2d = fig.add_subplot(grid[0, 1])

# ===============================================================
# === Left panel: the bundle
# ===============================================================
plot_manifolds.plot_manifold(
    ax=ax3d, manifold=bundle.total_space, resolution=140,
    xmin=-jnp.pi, xmax=jnp.pi, ymin=fibre_range[0], ymax=fibre_range[1],
    color=color_total_space, alpha=0.32, edgecolor="none", label=None
)

plot_bundles.plot_preimage_of_chart_domain(
    ax=ax3d, bundle=bundle, chart=chart_U, fibre_range=fibre_range,
    color="0.35", alpha=0.32, boundary_color=color_chart,
    label=r"$T\mathcal{U}$", label_offset=(0.12, 0.18, 0.18), label_fontsize=16
)

plot_bundles.plot_zero_section(
    ax=ax3d, bundle=bundle, param_range=(-jnp.pi, jnp.pi),
    color=color_base, linewidth=3.5, label=None
)

plot_bundles.plot_fibre(
    ax=ax3d, bundle=bundle, param_point=p_param, fibre_range=fibre_range,
    color=color_fibre, linewidth=1.4
)

plot_bundles.plot_projection_arrow(
    ax=ax3d, bundle=bundle, param_point=p_param, from_value=-0.85, to_value=-0.06,
    color=color_fibre, label=r"$\pi$", label_offset=(0.10, 0.10, -0.08), label_fontsize=15
)

plot_bundles.plot_section_on_total_space(
    ax=ax3d, bundle=bundle, field=field_X, param_range=(-jnp.pi, jnp.pi),
    color=color_field, linewidth=2.2, label=r"$\mathcal{X}$",
    label_position=0.30, label_offset=(0.0, 0.0, 0.22), label_fontsize=20
)

base_3d, tip_3d = plot_bundles.plot_vector_in_fibre(
    ax=ax3d, bundle=bundle, param_point=p_param, fibre_component=p_component,
    color=color_vector, base_point_color=color_fibre, label=r"$v_{\gamma, p}$",
    label_offset=(0.22, 0.05, -0.12), label_fontsize=15
)

ax3d.text(base_3d[0] + 0.05, base_3d[1] + 0.05, base_3d[2] - 0.22, r"$p$",
          fontsize=15, color=color_fibre)
ax3d.text2D(0.5, 0.94,
            r"$\left( T\mathcal{M}, \mathcal{O}_{T\mathcal{M}}, \mathcal{A}_{T\mathcal{M}} \right)_{2d}$",
            transform=ax3d.transAxes, fontsize=16, color="teal", ha="center")
ax3d.text2D(0.30, 0.06,
            r"$\left( \mathcal{M}, \mathcal{O}, \mathcal{A}_{C^\infty} \right)_{d}$",
            transform=ax3d.transAxes, fontsize=16, color=color_base, ha="center")

ax3d.view_init(elev=18.0, azim=28.0)
ax3d.set_box_aspect([1, 1, 1.15])

# ===============================================================
# === Right panel: the image of the bundle chart
# ===============================================================
component_range = (-1.1, 1.1)

x_min, x_max = plot_bundles.plot_bundle_chart_image(
    ax=ax2d, bundle=bundle, chart=chart_U, component_range=component_range,
    color=color_chart
)

plot_bundles.style_bundle_chart_axes(
    ax=ax2d, xlim=(-0.75, 1.05), ylim=(-1.45, 1.45),
    base_axis_label=r"$\mathbb{R}^d$", space_label=r"$\mathbb{R}^{2d}$"
)

plot_bundles.plot_section_in_bundle_chart(
    ax=ax2d, bundle=bundle, chart=chart_U, field=field_X,
    color=color_field, linewidth=2.2
)

base_2d, tip_2d = plot_bundles.plot_vector_in_bundle_chart(
    ax=ax2d, bundle=bundle, chart=chart_U, param_point=p_param,
    fibre_component=p_component, component_range=component_range,
    color=color_vector, fibre_color=color_fibre
)

ax2d.text(x_max + 0.04, component_range[1] - 0.05, r"$\xi_{\mathscr{X}}(T\mathcal{U})$",
          fontsize=14, color=color_chart, ha="left", va="top")

# ===============================================================
# === The maps, as arrows between the two panels
# ===============================================================
fig.canvas.draw()      # needed before projecting 3D points to figure coordinates

start_xi = plot_bundles.point_to_figure_fraction(ax3d, tip_3d)
end_xi = plot_bundles.point_to_figure_fraction(ax2d, tip_2d)
start_chart = plot_bundles.point_to_figure_fraction(ax3d, base_3d)
end_chart = plot_bundles.point_to_figure_fraction(ax2d, base_2d)

plot_bundles.add_map_arrow(
    fig, start_xi + np.array([0.015, 0.06]), end_xi + np.array([0.0, 0.07]),
    label=r"$\xi_{\mathscr{X}}$", rad=-0.32, label_position=0.45,
    label_offset=(0.0, 0.02), label_fontsize=20, color=color_chart
)
plot_bundles.add_map_arrow(
    fig, start_chart + np.array([0.012, -0.045]), end_chart + np.array([0.0, -0.045]),
    label=r"$\mathscr{X}$", rad=0.32, label_position=0.45,
    label_offset=(0.0, -0.02), label_fontsize=20, color=color_chart
)
plot_bundles.add_map_arrow(
    fig, start_xi + np.array([0.012, 0.0]), end_xi - np.array([0.012, 0.0]),
    label=r"$(d\mathscr{X}^1)_p$", rad=0.0, arrowstyle="-", linestyle=(0, (1, 2)),
    label_position=0.55, label_offset=(0.0, 0.022), label_fontsize=15, color=color_chart
)

plt.show()

fig.savefig("../saved_figures_and_videos/"+"Fig_tangent_bundle.pdf", bbox_inches="tight", dpi=300)